# 05 — European Options: Multi-Engine Pricing

Price a single European call option using **10+ engines** and compare:
- **BSM analytic** (Black-Scholes-Merton)
- **Black's formula**
- **Integral engine** (numerical integration)
- **Heston semi-analytic** (stochastic vol)
- **Finite differences** (Crank-Nicolson)
- **Binomial trees**: CRR, JR, Tian, Trigeorgis, Leisen-Reimer
- **Monte Carlo** (antithetic variates)

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare_table, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
QL = load_quantlib()

---
## Option Parameters

| Parameter | Value |
|---|---|
| Spot $S$ | 100 |
| Strike $K$ | 100 |
| Maturity $T$ | 1 year |
| Risk-free rate $r$ | 5% |
| Dividend yield $q$ | 2% |
| Volatility $\sigma$ | 20% |

In [ ]:
# Common parameters
S, K, T = 100.0, 100.0, 1.0
r, q, sigma = 0.05, 0.02, 0.20
CALL = 1   # OptionType.Call

---
## 1. QuantLib Reference Price

In [ ]:
ql_today = QL.Date(15, 1, 2024)
QL.Settings.instance().evaluationDate = ql_today

ql_spot   = QL.QuoteHandle(QL.SimpleQuote(S))
ql_r_ts   = QL.YieldTermStructureHandle(QL.FlatForward(ql_today, r, QL.Actual365Fixed()))
ql_q_ts   = QL.YieldTermStructureHandle(QL.FlatForward(ql_today, q, QL.Actual365Fixed()))
ql_vol_ts = QL.BlackVolTermStructureHandle(QL.BlackConstantVol(ql_today, QL.NullCalendar(), sigma, QL.Actual365Fixed()))

ql_process = QL.BlackScholesMertonProcess(ql_spot, ql_q_ts, ql_r_ts, ql_vol_ts)

exercise = QL.EuropeanExercise(ql_today + QL.Period(1, QL.Years))
payoff   = QL.PlainVanillaPayoff(QL.Option.Call, K)
ql_option = QL.VanillaOption(payoff, exercise)

# Analytic BSM
ql_option.setPricingEngine(QL.AnalyticEuropeanEngine(ql_process))
ql_bsm = ql_option.NPV()
print(f"QL BSM analytic: {ql_bsm:.10f}")

# Additional engines
ql_option.setPricingEngine(QL.IntegralEngine(ql_process))
ql_integral = ql_option.NPV()

ql_prices = {"BSM Analytic": ql_bsm, "Integral": ql_integral}

# Binomial trees
for name in ["crr", "jr", "tian", "trigeorgis", "lr"]:
    ql_option.setPricingEngine(QL.BinomialVanillaEngine(ql_process, name, 500))
    ql_prices[f"Binomial ({name})"] = ql_option.NPV()

# FD
ql_option.setPricingEngine(QL.FdBlackScholesVanillaEngine(ql_process, 200, 200))
ql_prices["FD Crank-Nicolson"] = ql_option.NPV()

# MC
ql_mc_engine = QL.MCEuropeanEngine(ql_process, "PseudoRandom",
                                    timeSteps=1, requiredSamples=100000, seed=42)
ql_option.setPricingEngine(ql_mc_engine)
ql_prices["Monte Carlo"] = ql_option.NPV()

print("\nQuantLib results:")
for name, price in ql_prices.items():
    print(f"  {name:25s}: {price:.8f}")

---
## 2. ql-jax Multi-Engine Pricing

In [ ]:
from ql_jax.engines.analytic.black_formula import black_scholes_price
from ql_jax.engines.analytic.heston import heston_price
from ql_jax.engines.analytic.integral import integral_price
from ql_jax.engines.fd.black_scholes import fd_black_scholes_price
from ql_jax.engines.lattice.binomial import binomial_price
from ql_jax.engines.mc.european import mc_european_bs

jax_prices = {}

# BSM Analytic
jax_prices["BSM Analytic"] = float(black_scholes_price(S, K, T, r, q, sigma, CALL))

# Integral
jax_prices["Integral"] = float(integral_price(S, K, T, r, q, sigma, CALL))

# Heston (with equivalent BSM parameters: v0=sigma^2, zero vol-of-vol)
jax_prices["Heston (≈BSM)"] = float(heston_price(
    S, K, T, r, q,
    v0=sigma**2, kappa=1.0, theta=sigma**2, xi=1e-6, rho=0.0,
    option_type=CALL))

# Finite Differences
jax_prices["FD Crank-Nicolson"] = float(fd_black_scholes_price(
    S, K, T, r, q, sigma, CALL, n_space=200, n_time=200))

# Binomial trees
for name in ["crr", "jr", "tian", "trigeorgis", "leisen_reimer"]:
    jax_prices[f"Binomial ({name})"] = float(binomial_price(
        S, K, T, r, q, sigma, CALL, n_steps=500, tree_type=name))

# Monte Carlo
mc_price, mc_err = mc_european_bs(S, K, T, r, q, sigma, CALL,
                                   n_paths=100_000, key=jax.random.PRNGKey(42))
jax_prices["Monte Carlo"] = float(mc_price)

print("ql-jax results:")
for name, price in jax_prices.items():
    print(f"  {name:25s}: {price:.8f}")

In [ ]:
# Master comparison table
bsm_exact = jax_prices["BSM Analytic"]

rows = []
for name in jax_prices:
    ql_val = ql_prices.get(name, ql_prices.get(name.replace("leisen_reimer", "lr"), None))
    if ql_val is not None:
        rows.append((name, ql_val, jax_prices[name]))

compare_table(rows, title="European Call: Multi-Engine Comparison")

In [ ]:
# ── Convergence: binomial steps ────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np

steps_list = [10, 20, 50, 100, 200, 500, 1000]
crr_prices = [float(binomial_price(S, K, T, r, q, sigma, CALL,
                                    n_steps=n, tree_type="crr"))
              for n in steps_list]
lr_prices = [float(binomial_price(S, K, T, r, q, sigma, CALL,
                                   n_steps=n, tree_type="leisen_reimer"))
             for n in steps_list]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(steps_list, crr_prices, 'o-', label='CRR')
ax.plot(steps_list, lr_prices, 's-', label='Leisen-Reimer')
ax.axhline(bsm_exact, color='r', linestyle='--', label=f'BSM exact = {bsm_exact:.6f}')
ax.set_xlabel('Number of Steps')
ax.set_ylabel('Option Price')
ax.set_title('Binomial Tree Convergence')
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

---
## 3. Quick Greeks via AD

In [ ]:
def bsm_call(spot, vol, rate, div):
    return black_scholes_price(spot, K, T, rate, div, vol, CALL)

args = (jnp.float64(S), jnp.float64(sigma), jnp.float64(r), jnp.float64(q))
greeks = jax.grad(bsm_call, argnums=(0, 1, 2, 3))(*args)

labels = ["Delta (dP/dS)", "Vega (dP/dσ)", "Rho (dP/dr)", "dP/dq"]
for label, g in zip(labels, greeks):
    print(f"  {label:20s}: {float(g):+.6f}")

# Gamma
gamma = jax.grad(jax.grad(bsm_call, argnums=0), argnums=0)(*args)
print(f"  {'Gamma (d²P/dS²)':20s}: {float(gamma):+.6f}")

---
## 4. Performance: Engine Comparison

In [ ]:
from notebooks._common import plot_speedup

engines_to_bench = [
    ("BSM",  lambda: black_scholes_price(S, K, T, r, q, sigma, CALL)),
    ("FD",   lambda: fd_black_scholes_price(S, K, T, r, q, sigma, CALL)),
    ("CRR 500", lambda: binomial_price(S, K, T, r, q, sigma, CALL, n_steps=500)),
]

ql_bench = [
    ("BSM",  lambda: (ql_option.setPricingEngine(QL.AnalyticEuropeanEngine(ql_process)), ql_option.NPV())[-1]),
    ("FD",   lambda: (ql_option.setPricingEngine(QL.FdBlackScholesVanillaEngine(ql_process, 200, 200)), ql_option.NPV())[-1]),
    ("CRR 500", lambda: (ql_option.setPricingEngine(QL.BinomialVanillaEngine(ql_process, "crr", 500)), ql_option.NPV())[-1]),
]

labels, ql_times, jax_times = [], [], []
for (name, jfn), (_, qfn) in zip(engines_to_bench, ql_bench):
    qt, _ = timed_ms(qfn, warmup=3, runs=10)
    jt, _ = timed_ms(jfn, warmup=3, runs=10)
    labels.append(name)
    ql_times.append(qt)
    jax_times.append(jt)
    print(f"  {name:10s}  QL={qt:.3f}ms  JAX={jt:.3f}ms  speedup={qt/jt:.1f}×")

plot_speedup(labels, ql_times, jax_times, title="Engine Benchmark")

---
## 5. Exercises

1. Add a **put** option and verify put-call parity: $C - P = S e^{-qT} - K e^{-rT}$.
2. Price the same option under Heston with realistic stochastic vol parameters ($\kappa=2$, $\theta=0.04$, $\xi=0.3$, $\rho=-0.7$, $v_0=0.04$). How does the price compare to BSM?
3. Build a **convergence table** for FD: vary `n_space` from 50 to 1000 and plot error vs grid size.

---
*End of Notebook 05*